In [ ]:
import pandas as pd
import requests
import json
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext

# FLASK/ Dont use names use p1, p2,

# ===============================================================
# COSTCO NIGHT MERCH SCHEDULER
# LLM + Deterministic Engine with Spreadsheet GUI
# ===============================================================

LLM_MODEL = "llama3"
LLM_URL = "http://localhost:11434/api/generate"

UNTRAINED = 0                 # 0 = untrained (penalized in avg_rank)
REQUIRED_ROLE_LINES = 15      # 6 drivers + 9 stockers

DAYS = ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY", "SUNDAY"]

# Map roles → skill columns in employee_skills.csv
ROLE_SKILL_MAP = {
    "DOCK": "dock",
    "EPJ": "epj",
    "WALLS": "walls",
    "DDD": "ddd",
    "FREEZER": "freezer_driver",
    "FREEZER_STOCKER": "freezer_stocker",
    "WALLS_STOCKER": "walls_stocker",
    "BEER": "beer_stocker",
}

SUPERVISOR_NAME = "Jacob"

rank_cols = [
    "dock", "epj", "walls", "ddd",
    "freezer_driver", "freezer_stocker",
    "walls_stocker", "beer_stocker",
]

# These are set via the GUI "Load CSV" buttons
skills_df = None
schedule_df = None


# ===============================================================
# LLM CALL
# ===============================================================

def run_local_llm(prompt: str) -> str:
    """Send a prompt to the local LLM and return its response text."""
    resp = requests.post(
        LLM_URL,
        json={"model": LLM_MODEL, "prompt": prompt, "stream": False}
    )
    resp.raise_for_status()
    return resp.json()["response"]


# ===============================================================
# DATA PROCESSING
# ===============================================================

def melt_schedule():
    """
    Convert weekly_schedule.csv from wide format:

        name | Monday | Tuesday | ...

    into long format:

        name | day | available
    """
    df = schedule_df.copy()
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={c: c.capitalize() for c in df.columns if c != "name"})
    return df.melt(id_vars=["name"], var_name="day", value_name="available")


def get_available(day, schedule_long):
    """Return list of employees available on the given ALL-CAPS day."""
    return schedule_long[
        (schedule_long["day"] == day.capitalize()) &
        (schedule_long["available"] == 1)
    ]["name"].tolist()


def sorted_available_df(day, schedule_long):
    """Return DataFrame of available employees sorted by average skill."""
    available = get_available(day, schedule_long)
    df = skills_df[skills_df["name"].isin(available)].copy()
    # Penalize untrained (0) as 10 for averaging
    df["avg_rank"] = df[rank_cols].replace(UNTRAINED, 10).mean(axis=1)
    return df.sort_values("avg_rank")


def best_candidates_for_day(day, schedule_long):
    """
    For each role, return the top 5 trained candidates.
    This is passed to the LLM as guidance.
    """
    df = sorted_available_df(day, schedule_long)

    def top(df_local, col):
        tmp = df_local[df_local[col] > 0][["name", col]]
        return tmp.sort_values(col).head(5).to_dict(orient="records")

    out = {}
    for role, col in ROLE_SKILL_MAP.items():
        out[role] = top(df, col)
    return out


# ===============================================================
# VALIDATION OF AI OUTPUT
# ===============================================================

def validate_day(day_block: str, available_today):
    """
    Validate LLM output for one day.

    • Must have exactly 15 lines of ROLE: NAME
    • All workers must be available
    • No worker appears twice
    """
    if not day_block.strip():
        return False

    lines = [l for l in day_block.splitlines() if ":" in l]
    if len(lines) != REQUIRED_ROLE_LINES:
        return False

    used = {}

    for line in lines:
        role, name = line.split(":", 1)
        name = name.strip()

        if name == "UNFILLED":
            continue

        if name not in available_today:
            return False

        used[name] = used.get(name, 0) + 1

    # Everyone must appear at most once
    return all(count == 1 for count in used.values())


# ===============================================================
# DETERMINISTIC FALLBACK SCHEDULER
# ===============================================================

def deterministic(day, schedule_long):
    """
    Fallback scheduler that always creates a valid schedule
    using skill rankings only (no AI).
    """
    available = get_available(day, schedule_long)
    df = skills_df[skills_df["name"].isin(available)].copy()
    df["avg_rank"] = df[rank_cols].replace(UNTRAINED, 10).mean(axis=1)
    df = df.sort_values("avg_rank")
    remaining = df.copy()

    def pick(col):
        """Pick best trained worker for a specific role."""
        nonlocal remaining
        tmp = remaining.sort_values(col)
        for _, r in tmp.iterrows():
            if r[col] > 0:
                name = r["name"]
                remaining = remaining[remaining["name"] != name]
                return name
        return "UNFILLED"

    def pick_avg():
        """Pick best remaining worker overall."""
        nonlocal remaining
        if remaining.empty:
            return "UNFILLED"
        name = remaining.iloc[0]["name"]
        remaining = remaining[remaining["name"] != name]
        return name

    out = []
    out.append(f"=== {day} ===")
    out.append("DRIVERS")
    out.append(f"DOCK: {pick('dock')}")
    out.append(f"EPJ: {pick('epj')}")
    out.append(f"WALLS: {pick('walls')}")
    out.append(f"FREEZER: {pick('freezer_driver')}")
    out.append(f"DDD: {pick('ddd')}")
    out.append(f"FLOAT: {pick_avg()}")

    out.append("\nSTOCKERS")
    for _ in range(5):
        out.append(f"FREEZER: {pick('freezer_stocker')}")
    for _ in range(3):
        out.append(f"WALLS: {pick('walls_stocker')}")
    out.append(f"BEER: {pick('beer_stocker')}")

    return "\n".join(out), remaining["name"].tolist()


# ===============================================================
# HYBRID SCHEDULER (LLM + FALLBACK)
# ===============================================================

def run_scheduler():
    """
    Run the full weekly scheduler:
      • Ask LLM for schedule
      • Validate; if invalid → deterministic fallback
      • Append EXTRAS list for each day
      • Return text representation for the whole week
    """
    schedule_long = melt_schedule()

    # Example candidates for the LLM to learn from
    example = json.dumps(best_candidates_for_day("WEDNESDAY", schedule_long), indent=2)

    template = """
DRIVERS
DOCK: ______
EPJ: ______
WALLS: ______
FREEZER: ______
DDD: ______
FLOAT: ______

STOCKERS
FREEZER: ______
FREEZER: ______
FREEZER: ______
FREEZER: ______
FREEZER: ______
WALLS: ______
WALLS: ______
WALLS: ______
BEER: ______
"""

    prompt = f"""
You are an expert Costco night merch scheduler.
Generate a weekly schedule for Monday–Sunday.
Use only available workers. No duplicates. Avoid untrained unless needed.

Example best candidates for Wednesday:
{example}

TEMPLATES:
""" + "\n".join([f"=== {day} ===\n{template}" for day in DAYS])

    try:
        llm_output = run_local_llm(prompt)
    except Exception:
        llm_output = ""

    # ---------- Parse the LLM output into per-day blocks ----------
    day_blocks = {}
    cur_day = None
    cur_text = []

    for line in llm_output.splitlines():
        s = line.strip()
        if s.startswith("===") and s.endswith("==="):
            if cur_day:
                day_blocks[cur_day] = "\n".join(cur_text)
            cur_day = s.replace("=", "").strip()  # e.g. "MONDAY"
            cur_text = [s]
        else:
            cur_text.append(line)

    if cur_day:
        day_blocks[cur_day] = "\n".join(cur_text)

    # ---------- Validate or fall back for each day ----------
    final_lines = []

    for day in DAYS:
        avail = get_available(day, schedule_long)
        block = day_blocks.get(day, "")

        if block and validate_day(block, avail):
            final_lines.append(block)
            extras = []
        else:
            block, extras = deterministic(day, schedule_long)
            final_lines.append(block)

        final_lines.append("EXTRAS: " + (", ".join(extras) if extras else "NONE"))
        final_lines.append("")

    return "\n".join(final_lines)


# ===============================================================
# SPREADSHEET-STYLE GUI RENDERING
# ===============================================================

def display_schedule_table(schedule_text: str):
    """
    Parse the text schedule and render it as a table that looks
    like a Google Sheet:
      • Days = columns
      • Roles = rows
      • DRIVERS / STOCKERS sections with headers
      • Multiple FREEZER/WALLS stockers shown as separate rows
        (FREEZER 1, FREEZER 2, etc.)
    """

    # Clear any existing table widgets
    for widget in table_frame.winfo_children():
        widget.destroy()

    # parsed[DAY][FULL_KEY] = worker
    parsed = {d: {} for d in DAYS}
    role_order = []  # Ordered list of row keys

    current_day = None
    current_section = None
    freezer_count = 0
    walls_count = 0

    for line in schedule_text.splitlines():
        line = line.strip()
        if not line:
            continue

        # New day block: "=== MONDAY ==="
        if line.startswith("===") and line.endswith("==="):
            current_day = line.replace("=", "").strip()
            freezer_count = 0
            walls_count = 0
            continue

        # Section headers
        if line in ["DRIVERS", "STOCKERS"]:
            current_section = line
            freezer_count = 0
            walls_count = 0
            continue

        # EXTRAS: Colbrun, Mason, ...
        if line.startswith("EXTRAS:") and current_day:
            worker_list = line.split(":", 1)[1].strip()
            full_key = "STOCKERS_EXTRAS"  # treat extras like part of stockers section
            if full_key not in role_order:
                role_order.append(full_key)
            parsed[current_day][full_key] = worker_list
            continue

        # Regular "ROLE: NAME" lines
        if ":" in line and current_day and current_section:
            role, worker = line.split(":", 1)
            role = role.strip()
            worker = worker.strip()

            # Enumerate repeated stocker roles
            if current_section == "STOCKERS":
                if role == "FREEZER":
                    freezer_count += 1
                    role_key = f"FREEZER_{freezer_count}"
                elif role == "WALLS":
                    walls_count += 1
                    role_key = f"WALLS_{walls_count}"
                else:
                    # BEER or anything else
                    role_key = role
            else:
                # Drivers (each role appears once)
                role_key = role

            full_key = f"{current_section}_{role_key}"

            if full_key not in role_order:
                role_order.append(full_key)

            parsed[current_day][full_key] = worker

    # ---------- Build the grid UI ----------

    header_bg = "#f7ea48"   # yellow header like your sheet
    section_bg = "#666666"  # DRIVERS/STOCKERS header row
    driver_bg = "#dddddd"
    stocker_bg = "#cfd6ff"
    cell_bg = "#222222"
    text_color = "#ffffff"

    # Day header row
    tk.Label(
        table_frame, text="", width=14, bg=cell_bg
    ).grid(row=0, column=0, sticky="nsew")

    for col, day in enumerate(DAYS):
        tk.Label(
            table_frame,
            text=day,
            bg=header_bg,
            fg="black",
            font=("Arial", 11, "bold"),
            width=14,
            borderwidth=2,
            relief="solid",
        ).grid(row=0, column=col + 1, sticky="nsew")

    row_index = 1
    last_group = None

    for full_key in role_order:
        group, role_key = full_key.split("_", 1)  # e.g. DRIVERS_DOCK, STOCKERS_FREEZER_1

        # Insert group header row when switching from DRIVERS to STOCKERS
        if group != last_group:
            tk.Label(
                table_frame,
                text=group,
                bg=section_bg,
                fg=text_color,
                font=("Arial", 11, "bold"),
                borderwidth=2,
                relief="solid",
                width=14,
            ).grid(row=row_index, column=0, columnspan=len(DAYS)+1, sticky="nsew")
            row_index += 1
            last_group = group

        # Human-friendly role label on left
        if group == "STOCKERS":
            if role_key.startswith("FREEZER_"):
                idx = role_key.split("_")[1]
                role_label = f"FREEZER {idx}"
            elif role_key.startswith("WALLS_"):
                idx = role_key.split("_")[1]
                role_label = f"WALLS {idx}"
            else:
                role_label = role_key  # BEER, EXTRAS
        else:
            role_label = role_key  # DOCK, EPJ, etc.

        tk.Label(
            table_frame,
            text=role_label,
            bg=driver_bg if group == "DRIVERS" else stocker_bg,
            fg="black",
            font=("Arial", 10, "bold"),
            borderwidth=2,
            relief="solid",
            width=14,
        ).grid(row=row_index, column=0, sticky="nsew")

        # Fill each day cell
        for col, day in enumerate(DAYS):
            name = parsed[day].get(full_key, "")
            tk.Label(
                table_frame,
                text=name,
                bg=cell_bg,
                fg=text_color,
                font=("Arial", 10),
                borderwidth=1,
                relief="solid",
                width=14,
            ).grid(row=row_index, column=col + 1, sticky="nsew")

        row_index += 1


# ===============================================================
# TKINTER FRONT-END CONTROLS
# ===============================================================

def load_skills():
    """Prompt user to pick employee_skills.csv."""
    global skills_df
    path = filedialog.askopenfilename(title="Load employee_skills.csv")
    if path:
        skills_df = pd.read_csv(path)
        skills_df["name"] = skills_df["name"].str.strip()
        messagebox.showinfo("Loaded", "Employee skills CSV loaded.")


def load_schedule():
    """Prompt user to pick weekly_schedule.csv."""
    global schedule_df
    path = filedialog.askopenfilename(title="Load weekly_schedule.csv")
    if path:
        schedule_df = pd.read_csv(path)
        schedule_df["name"] = schedule_df["name"].str.strip()
        messagebox.showinfo("Loaded", "Weekly schedule CSV loaded.")


def run_and_display():
    """Run the scheduler and display the result as a table."""
    if skills_df is None or schedule_df is None:
        messagebox.showerror("Error", "Load both CSV files first.")
        return

    result = run_scheduler()

    # Keep text output in hidden box so Save Output works
    output_box.delete("1.0", tk.END)
    output_box.insert(tk.END, result)

    # Show table
    table_frame.pack(fill="both", expand=True, padx=10, pady=10)
    display_schedule_table(result)


def save_output():
    """Save raw text schedule to a .txt file."""
    text = output_box.get("1.0", tk.END)
    path = filedialog.asksaveasfilename(defaultextension=".txt")
    if path:
        with open(path, "w") as f:
            f.write(text)
        messagebox.showinfo("Saved", "Schedule saved.")


# ===============================================================
# BUILD GUI
# ===============================================================

root = tk.Tk()
root.title("Costco Night Merch Scheduler")
root.geometry("1400x900")
root.configure(bg="#333333")

top_frame = tk.Frame(root, bg="#333333")
top_frame.pack(pady=10)

tk.Button(top_frame, text="Load Skills CSV", width=20, command=load_skills).grid(row=0, column=0, padx=10)
tk.Button(top_frame, text="Load Weekly Schedule CSV", width=25, command=load_schedule).grid(row=0, column=1, padx=10)
tk.Button(top_frame, text="Run Scheduler", width=20, command=run_and_display).grid(row=0, column=2, padx=10)
tk.Button(top_frame, text="Save Output (Text)", width=20, command=save_output).grid(row=0, column=3, padx=10)

# Frame where the spreadsheet-style table will be drawn
table_frame = tk.Frame(root, bg="#333333")
table_frame.pack(fill="both", expand=True, padx=10, pady=10)

# Hidden text box used only for saving raw output
output_box = scrolledtext.ScrolledText(root, font=("Courier", 10))
output_box.pack_forget()

root.mainloop()


2025-12-09 09:13:28.894 python[61890:35223331] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'
